# Lesson Trino

Урок лучше проходить сверху вниз: каждая маленькая ячейка показывает один шаг и сразу даёт проверочный вывод. Здесь Trino выступает как SQL-движок поверх разных catalog: `iceberg`, `postgresql`, `clickhouse`, `greenplum` и служебного `jmx_monitor`.


## Что такое Trino catalog, schema и table

- `iceberg` - это catalog Trino, подключённый к Iceberg REST catalog.
- `NAMESPACE` - это ваша личная schema внутри catalog `iceberg`. Впишите туда свой Telegram-ник или фамилию латиницей.
- `orders` или `events_partition_demo` - это table внутри namespace.
- Физически Iceberg-файлы лежат в MinIO bucket `lakehouse`.
- Trino не хранит данные сам: он читает и пишет через catalog/connectors.


## 1. Подключение к Trino

В этом notebook используем HTTP API Trino напрямую через `requests`, чтобы не ставить отдельный Python-пакет `trino`. Из контейнера Jupyter Trino доступен как `http://trino:8080`. Если запускаете код с хоста, используйте `http://localhost:8085`.


In [1]:
import os
import time
import requests
import pandas as pd
from IPython.display import display

TRINO_URL = os.getenv("TRINO_URL", "http://trino:8080")
TRINO_USER = os.getenv("TRINO_USER", "jupyter")

print("TRINO_URL =", TRINO_URL)
print("TRINO_USER =", TRINO_USER)


TRINO_URL = http://trino:8080
TRINO_USER = jupyter


Helper ниже отправляет SQL в Trino, ждёт завершения запроса и возвращает результат как pandas DataFrame. Если Trino вернёт ошибку, ячейка покажет текст ошибки из Trino.


In [2]:
def trino_query(sql_text: str, catalog: str | None = None, schema: str | None = None) -> pd.DataFrame:
    headers = {"X-Trino-User": TRINO_USER}
    if catalog:
        headers["X-Trino-Catalog"] = catalog
    if schema:
        headers["X-Trino-Schema"] = schema

    response = requests.post(
        f"{TRINO_URL}/v1/statement",
        data=sql_text.encode("utf-8"),
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()

    columns = payload.get("columns")
    rows = payload.get("data", [])

    while payload.get("nextUri"):
        response = requests.get(payload["nextUri"], timeout=30)
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            error = payload["error"]
            message = error.get("message", "Unknown Trino error")
            error_name = error.get("errorName", "TRINO_ERROR")
            raise RuntimeError(f"{error_name}: {message}")

        columns = columns or payload.get("columns")
        rows.extend(payload.get("data", []))

    if "error" in payload:
        error = payload["error"]
        message = error.get("message", "Unknown Trino error")
        error_name = error.get("errorName", "TRINO_ERROR")
        raise RuntimeError(f"{error_name}: {message}")

    if not columns:
        return pd.DataFrame(rows)

    column_names = [column["name"] for column in columns]
    return pd.DataFrame(rows, columns=column_names)


def show(sql_text: str, catalog: str | None = None, schema: str | None = None) -> pd.DataFrame:
    df = trino_query(sql_text, catalog=catalog, schema=schema)
    display(df)
    return df


Проверим, что Trino отвечает и какие catalog доступны.


In [3]:
show('SELECT * FROM clickhouse.gold.launches_by_country_day')
show('SELECT * FROM clickhouse.gold.rocket_success_rate')
show('SELECT * FROM clickhouse.gold.agency_top_10')

,launch_date,country,launches_count
0,2026-10-15,Q0E=,1


,rocket_name,total_launches,successful,failed,success_rate
0,RmFsY29uIDk=,1,0,0,0.0


,provider_name,launches_count
0,U3BhY2VY,1


,provider_name,launches_count
0,U3BhY2VY,1


In [ ]:
show("SHOW CATALOGS")


In [ ]:
show("SHOW SCHEMAS FROM iceberg")

In [ ]:
show("SHOW TABLES FROM iceberg.halltape")

In [ ]:
show("SELECT * FROM iceberg.halltape.orders ORDER BY order_id")

In [ ]:
show("SELECT COUNT(*) FROM iceberg.silver.launches")
show('SELECT file_path, record_count FROM iceberg.silver."launches$files"')

## 2. Рабочий namespace

❗️ В `NAMESPACE` обязательно впишите свой Telegram-ник или фамилию латиницей, например `NAMESPACE = "ivanov"`. Так каждый создаёт свои таблицы в отдельном namespace и не мешает другим.


In [ ]:
CATALOG = "iceberg"

# ❗️ Замените на свой Telegram-ник или фамилию латиницей, например: NAMESPACE = "ivanov".
# ❗️ Так у каждого будет свой namespace: iceberg.<ваш_ник>.
NAMESPACE = "bobr"

show(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{NAMESPACE}")
show(f"SHOW SCHEMAS FROM {CATALOG}")


### Если нужно полностью очистить свой namespace

Эту ячейку запускайте только если хотите удалить все учебные таблицы из своего namespace и начать урок заново. В Trino используем обычный `DROP TABLE`; для Iceberg таблиц Trino удаляет таблицу через Iceberg catalog.


In [ ]:
tables_df = show(f"SHOW TABLES FROM {CATALOG}.{NAMESPACE}")

for table_name in tables_df.get("Table", []):
    show(f"DROP TABLE IF EXISTS {CATALOG}.{NAMESPACE}.{table_name}")

show(f"SHOW TABLES FROM {CATALOG}.{NAMESPACE}")


## 3. Создание Iceberg-таблицы через Trino

Создадим таблицу `orders` в Iceberg catalog. В Trino свойства Iceberg задаются через `WITH (...)`: формат файлов, версия Iceberg table format и partitioning.


In [ ]:
show(f"""
CREATE TABLE {CATALOG}.{NAMESPACE}.orders (
    order_id INTEGER,
    customer VARCHAR,
    city VARCHAR,
    category VARCHAR,
    amount DOUBLE,
    event_ts TIMESTAMP(6)
)
WITH (
    format = 'PARQUET',
    format_version = 2,
    partitioning = ARRAY['day(event_ts)']
)
""")


Посмотрим, как Trino видит созданную таблицу.


In [ ]:
show(f"SHOW TABLES FROM {CATALOG}.{NAMESPACE}")
show(f"DESCRIBE {CATALOG}.{NAMESPACE}.orders")


## 4. INSERT и чтение данных

Добавим несколько строк. После insert Iceberg создаст snapshot и физические parquet-файлы в MinIO: `lakehouse/<ваш_ник>/orders/...`.


In [ ]:
show(f"""
INSERT INTO {CATALOG}.{NAMESPACE}.orders VALUES
    (1001, 'Alice', 'Moscow', 'electronics', 1299.90, TIMESTAMP '2026-04-10 10:15:00'),
    (1002, 'Bob', 'Kazan', 'books', 42.50, TIMESTAMP '2026-04-10 12:05:00'),
    (1003, 'Carrol', 'Saint Petersburg', 'home', 310.00, TIMESTAMP '2026-04-11 09:30:00'),
    (1004, 'Diana', 'Sochi', 'sport', 560.40, TIMESTAMP '2026-04-11 19:45:00'),
    (1005, 'Egor', 'Moscow', 'games', 74.99, TIMESTAMP '2026-04-12 14:20:00'),
    (1006, 'Fatima', 'Kazan', 'electronics', 2200.00, TIMESTAMP '2026-04-12 16:00:00')
""")

show(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")


## 5. Что физически появилось в S3

После первой записи зайдите в MinIO bucket `lakehouse` и откройте папку своего namespace: `lakehouse/<ваш_ник>/orders/`. Там будут `data/` с parquet-файлами и `metadata/` с metadata-файлами Iceberg.

В Trino обычно не читают произвольный parquet/avro/json по S3-пути напрямую. Для изучения физического слоя просто откройте файлы в MinIO или используйте metadata tables ниже.


In [ ]:
print("Примеры путей, которые можно найти в MinIO:")
print(f"s3://lakehouse/{NAMESPACE}/orders/data/...")
print(f"s3://lakehouse/{NAMESPACE}/orders/metadata/...")


## 6. Metadata tables Iceberg в Trino

Trino показывает служебную информацию Iceberg через metadata tables. Их имена пишутся в кавычках: `"orders$snapshots"`, `"orders$history"`, `"orders$files"`, `"orders$partitions"`, `"orders$manifests"`.


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."orders$history"
ORDER BY made_current_at
""")


In [ ]:
show(f"""
SELECT content, file_path, record_count, file_size_in_bytes
FROM {CATALOG}.{NAMESPACE}."orders$files"
ORDER BY file_path
""")


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."orders$partitions"
""")


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."orders$manifests"
""")

## 7. Append и новые snapshots

Каждая запись данных или изменение metadata создаёт новый snapshot. Добавим ещё строки и посмотрим, как изменилась история.


In [ ]:
show(f"""
INSERT INTO {CATALOG}.{NAMESPACE}.orders VALUES
    (1007, 'Gleb', 'Novosibirsk', 'books', 23.90, TIMESTAMP '2026-04-13 11:10:00'),
    (1008, 'Hannah', 'Moscow', 'home', 415.30, TIMESTAMP '2026-04-13 18:25:00'),
    (1009, 'Ivan', 'Vladivostok', 'travel', 980.00, TIMESTAMP '2026-04-14 08:40:00')
""")

show(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")
show(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")


## 8. Time travel

Trino умеет читать Iceberg-таблицу на конкретный snapshot через `FOR VERSION AS OF`. Это только чтение старой версии: текущая таблица не меняется.


In [ ]:
snapshots_df = show(f"""
SELECT committed_at, snapshot_id
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")

first_snapshot_id = int(snapshots_df.iloc[0]["snapshot_id"])
latest_snapshot_id = int(snapshots_df.iloc[-1]["snapshot_id"])

print("first_snapshot_id =", first_snapshot_id)
print("latest_snapshot_id =", latest_snapshot_id)


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}.orders FOR VERSION AS OF {first_snapshot_id}
ORDER BY order_id
""")


In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}.orders FOR VERSION AS OF {latest_snapshot_id}
ORDER BY order_id
""")


## 9. DELETE и snapshots

Удалим часть строк. В Iceberg это создаёт новый snapshot. В зависимости от режима записи и connector-реализации физически могут появляться новые data/delete files.


In [ ]:
show(f"DELETE FROM {CATALOG}.{NAMESPACE}.orders WHERE amount < 100")

show(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")
show(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")
show(f"""
SELECT content, count(*) AS files, sum(record_count) AS records
FROM {CATALOG}.{NAMESPACE}."orders$files"
GROUP BY content
ORDER BY content
""")


## 10. Schema evolution

Iceberg хранит схему в metadata. Через Trino можно добавлять, переименовывать и удалять колонки без ручной перезаписи всех parquet-файлов.


In [ ]:
show(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders ADD COLUMN priority VARCHAR")
show(f"UPDATE {CATALOG}.{NAMESPACE}.orders SET priority = 'normal' WHERE priority IS NULL")
show(f"SELECT order_id, customer, amount, priority FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")


In [ ]:
show(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders RENAME COLUMN customer TO customer_name")
show(f"DESCRIBE {CATALOG}.{NAMESPACE}.orders")
show(f"SELECT order_id, customer_name, amount, priority FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")


Удалим учебную колонку. Логическая схема изменится, а старые физические файлы не обязаны сразу переписываться.


In [ ]:
show(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders DROP COLUMN priority")
show(f"DESCRIBE {CATALOG}.{NAMESPACE}.orders")


## 11. Partition evolution

Iceberg умеет жить с несколькими partition specs одновременно. Старые файлы остаются со старой spec, а новые данные после изменения partitioning пишутся уже по новой spec.

В Trino для Iceberg partition evolution задаётся через изменение table property `partitioning`.


In [ ]:
show(f"""
CREATE TABLE {CATALOG}.{NAMESPACE}.events_partition_demo (
    event_id INTEGER,
    user_id INTEGER,
    category VARCHAR,
    event_ts TIMESTAMP(6)
)
WITH (
    format = 'PARQUET',
    format_version = 2,
    partitioning = ARRAY['day(event_ts)']
)
""")

show(f"""
INSERT INTO {CATALOG}.{NAMESPACE}.events_partition_demo VALUES
    (1, 10, 'search', TIMESTAMP '2026-04-10 10:00:00'),
    (2, 10, 'view', TIMESTAMP '2026-04-10 10:01:00'),
    (3, 11, 'cart', TIMESTAMP '2026-04-11 12:30:00'),
    (4, 12, 'purchase', TIMESTAMP '2026-04-12 18:45:00')
""")

show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."events_partition_demo$partitions"
""")
show(f"""
SELECT spec_id, content, file_path, record_count
FROM {CATALOG}.{NAMESPACE}."events_partition_demo$files"
ORDER BY file_path
""")


Добавим новую partition field `category`. Старые файлы не раскладываются заново; новая spec применяется к новым записям.


In [ ]:
show(f"""
ALTER TABLE {CATALOG}.{NAMESPACE}.events_partition_demo
SET PROPERTIES partitioning = ARRAY['day(event_ts)', 'category']
""")

show(f"""
INSERT INTO {CATALOG}.{NAMESPACE}.events_partition_demo VALUES
    (5, 13, 'search', TIMESTAMP '2026-04-13 09:00:00'),
    (6, 13, 'purchase', TIMESTAMP '2026-04-13 09:20:00')
""")

show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}."events_partition_demo$partitions"
""")
show(f"""
SELECT spec_id, content, file_path, record_count
FROM {CATALOG}.{NAMESPACE}."events_partition_demo$files"
ORDER BY spec_id, file_path
""")


## 12. EXPLAIN и фильтрация

Trino показывает план запроса через `EXPLAIN`. Для Iceberg важно смотреть, какие фильтры попадают в table scan: так видно, что connector может использовать metadata и partition pruning.


In [ ]:
plan = show(f"""
EXPLAIN
SELECT *
FROM {CATALOG}.{NAMESPACE}.events_partition_demo
WHERE event_ts >= TIMESTAMP '2026-04-10 00:00:00'
  AND event_ts < TIMESTAMP '2026-04-11 00:00:00'
""")


In [ ]:
plan_text = plan.iloc[0]["Query Plan"]
print(plan_text)

In [ ]:
show(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}.events_partition_demo
WHERE event_ts >= TIMESTAMP '2026-04-10 00:00:00'
  AND event_ts < TIMESTAMP '2026-04-11 00:00:00'
ORDER BY event_id
""")


## 13. Maintenance: optimize и expire snapshots

Trino умеет запускать Iceberg maintenance-команды через `ALTER TABLE ... EXECUTE`. Они полезны для compact/rewrite файлов и очистки старых snapshots. В учебном ноутбуке команды ниже оставлены отдельными ячейками, чтобы было видно результат.


In [ ]:
show(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders EXECUTE optimize")


In [ ]:
show(f"""
SELECT content, count(*) AS files, sum(record_count) AS records
FROM {CATALOG}.{NAMESPACE}."orders$files"
GROUP BY content
ORDER BY content
""")


In [ ]:
show(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id")

`expire_snapshots` удаляет старые snapshots с учётом retention. В реальных проектах это делают осторожно, потому что после очистки старые версии могут стать недоступны для time travel.


In [ ]:
# Пример команды. Раскомментируйте, если хотите явно потренировать очистку snapshots.
show(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders EXECUTE expire_snapshots(retention_threshold => '1h')")

show(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")


## 14. Работа с другими catalog

Trino может одним SQL-клиентом смотреть в разные источники. Ниже только обзорные команды: они помогают понять, какие schemas и tables доступны в текущем стенде.


In [ ]:
show("SHOW SCHEMAS FROM postgresql")
show("SHOW SCHEMAS FROM clickhouse")
show("SHOW SCHEMAS FROM greenplum")
show("SHOW SCHEMAS FROM jmx_monitor")


## 15. Что проверить в конце

После всех операций полезно посмотреть список таблиц, snapshots и физические файлы. Если в MinIO открыть `lakehouse/<ваш_ник>/...`, там будут видны data-файлы и metadata-файлы Iceberg.


In [ ]:
show(f"SHOW TABLES FROM {CATALOG}.{NAMESPACE}")
show(f"""
SELECT committed_at, snapshot_id, operation
FROM {CATALOG}.{NAMESPACE}."orders$snapshots"
ORDER BY committed_at
""")
show(f"""
SELECT content, count(*) AS files, sum(record_count) AS records
FROM {CATALOG}.{NAMESPACE}."orders$files"
GROUP BY content
ORDER BY content
""")


In [ ]:
show("SHOW CATALOGS")

In [ ]:
import requests
r = requests.get("http://trino:8080/v1/info")
print(r.json())